## Results report (clean)\n\nLoads trained artifacts under `results/` and produces:\n- ECE-over-time plot\n- compact metrics table\n

In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nREPO_DIR = os.environ.get("REPO_DIR", "/content/Calibration-Confidence")\nif Path(REPO_DIR).is_dir():\n    os.chdir(REPO_DIR)\nelse:\n    os.chdir(Path.cwd().parent)\n\nROOT = os.path.abspath(os.getcwd())\nif ROOT not in sys.path:\n    sys.path.insert(0, ROOT)\n\nos.makedirs("results/figures", exist_ok=True)\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nprint("Repo root:", os.getcwd())\nprint("Device:", device)\n

In [ ]:
import glob\n\nprint("Checkpoints:")\nfor p in sorted(glob.glob("results/checkpoints/*.pt")):\n    print(" -", p)\n\nprint("\nResults (.npz):")\nfor p in sorted(glob.glob("results/*_results.npz")):\n    print(" -", p)\n

In [ ]:
from experiments.plot_ece_over_time import plot_ece_over_time\n\nnpz_paths = [\n    "results/mlp_results.npz",\n    "results/deep_results.npz",\n    "results/rnn_results.npz",\n    "results/lstm_results.npz",\n    "results/residual_results.npz",\n]\nlabels = ["mlp", "deep", "rnn", "lstm", "residual"]\n\nplot_ece_over_time(npz_paths=npz_paths, labels=labels, save_path="results/figures/ece_over_time_all.png")\n

In [ ]:
import os\nimport numpy as np\n\ntry:\n    import pandas as pd\n    PANDAS_AVAILABLE = True\nexcept Exception:\n    pd = None\n    PANDAS_AVAILABLE = False\n\nruns = [\n    ("mlp", "results/mlp_results.npz"),\n    ("deep", "results/deep_results.npz"),\n    ("rnn", "results/rnn_results.npz"),\n    ("lstm", "results/lstm_results.npz"),\n    ("residual", "results/residual_results.npz"),\n]\n\nrows = []\nfor name, path in runs:\n    if not os.path.isfile(path):\n        rows.append({"model": name, "path": path, "status": "missing"})\n        continue\n    d = np.load(path)\n    row = {"model": name, "path": path, "status": "ok"}\n    if "val_loss" in d:\n        row["final_val_loss"] = float(np.asarray(d["val_loss"])[-1])\n    if "ece_over_time" in d:\n        row["final_ece"] = float(np.asarray(d["ece_over_time"])[-1])\n    if "per_example_loss" in d:\n        row["mean_per_example_loss"] = float(np.mean(np.asarray(d["per_example_loss"]).ravel()))\n    rows.append(row)\n\nif PANDAS_AVAILABLE:\n    df = pd.DataFrame(rows)\n    cols = [c for c in ["model", "status", "final_val_loss", "final_ece", "mean_per_example_loss", "path"] if c in df.columns]\n    display(df[cols])\nelse:\n    for r in rows:\n        print(r)\n